# Complete Hub Prioritization Pipeline
## Grouping, Filtering, and Scoring

This notebook implements the complete hub prioritization framework:
1. Data loading and cleaning
2. Hub grouping (already done in previous step)
3. Eligibility filtering
4. Hub type classification
5. Scoring (5 criteria)
6. Monte Carlo aggregation
7. Results export

Based on: `Group_n_Filter_Hubs.ipynb` and `HubsScoring_vAugust2025.ipynb`

## Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Setup paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
RESULTS_DIR = DATA_DIR / "results"

# Create directories if needed
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Load config constants (from src/config.py)
TIER_NATIONAL = "ארצי"
TIER_METRO = "מטרופוליני"
TIER_LOCAL = "עירוני"

MODE_WEIGHTS = {
    'Funicular': 1.0,
    'Cable Line': 2.0,
    'BRT': 3.0,
    'LRT': 4.0,
    'Metro': 5.0,
    'Suburban Rail': 6.0,
    'Interurban Rail': 7.0,
    'HighSpeed Rail': 8.0,
    'Rail': 7.0,
    'Express Bus': 3.0,
    'Bus': 2.0,
}

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")

## Step 1: Load Grouped Hub Data

Load the output from the grouping step (from `grouped_hubs_ready_for_scoring_21082025.csv`)

In [ ]:
# Load grouped hubs data
# Update this path to your actual data file
input_file = PROCESSED_DATA_DIR / 'grouped_hubs_for_filtering.csv'

# Or load from the sample provided:
# input_file = PROJECT_ROOT / 'notebooks' / 'grouped_hubs_ready_for_scoring_21082025.csv'

df = pd.read_csv(input_file, encoding='utf-8')

print(f"Loaded {len(df)} grouped hubs")
df.head()

## Step 2: Data Cleaning & Preparation

Convert string representations to proper Python objects (from notebook logic)

In [ ]:
def split_list_string(txt):
    """Convert string representation of list to actual list."""
    if pd.isna(txt) or txt == '':
        return []
    txt = str(txt).replace("'", "").replace("[", "").replace("]", "").replace('"', "")
    return [x.strip() for x in txt.split(',') if x.strip()]

def split_node_list(txt):
    """Convert string representation of node list to integer list."""
    if pd.isna(txt) or txt == '':
        return []
    txt = str(txt).replace("'", "").replace("[", "").replace("]", "").replace('"', "").replace(',', "")
    return [int(x) for x in txt.split() if x.strip().isdigit()]

# Convert list columns
if 'Line_Unique' in df.columns:
    df['Line_Unique'] = df['Line_Unique'].apply(split_list_string)
    
if 'Mode_Planned' in df.columns:
    df['Mode_Planned'] = df['Mode_Planned'].apply(split_list_string)
    
if 'node' in df.columns:
    df['node'] = df['node'].apply(split_node_list)
    
if 'h3_index' in df.columns:
    df['h3_index'] = df['h3_index'].apply(split_list_string)

print("✓ Data cleaned and converted")
df[['Mode_Planned', 'Line_Unique', 'TotalDemand']].head()

## Step 3: Fix Mode Names

Clean up mode names to match expected format

In [ ]:
def correct_mode_planned(modes):
    """Clean up mode names and standardize."""
    if not isinstance(modes, list):
        return []
    
    cleaned = []
    for mode in modes:
        mode = mode.strip().replace(',', '')
        
        # Handle 'Rail' subcategories
        if mode == 'HighSpeed':
            cleaned.append('HighSpeed Rail')
        elif mode == 'Interurban':
            cleaned.append('Interurban Rail')
        elif mode == 'Suburban':
            cleaned.append('Suburban Rail')
        elif mode != 'Rail':  # Skip generic 'Rail' if we have specific types
            cleaned.append(mode)
    
    # Remove 'Rail' if we have specific rail types
    if 'Rail' in cleaned:
        specific_rails = ['HighSpeed Rail', 'Interurban Rail', 'Suburban Rail']
        if any(r in cleaned for r in specific_rails):
            cleaned = [m for m in cleaned if m != 'Rail']
    
    return cleaned

df['Mode_Planned'] = df['Mode_Planned'].apply(correct_mode_planned)

print("✓ Mode names corrected")
print("\nUnique modes found:")
all_modes = set()
for modes in df['Mode_Planned']:
    all_modes.update(modes)
print(sorted(all_modes))

## Step 4: Add Categorical Scores

Following the original notebook logic

In [ ]:
# Region category (area field)
def get_region_category(area):
    """Map area to region category (0 for Tel Aviv, 1 for others)."""
    if pd.isna(area):
        return 0
    area = str(area).strip()
    if 'תל' in area or 'Tel Aviv' in area:
        return 0
    return 1

# Location category (location field)
def get_location_category(location):
    """Map location to metro position category."""
    if pd.isna(location):
        return 1
    location = str(location).strip()
    if 'גלעין' in location:
        return 3  # Core
    elif 'טבעת' in location:
        return 2  # Ring
    else:
        return 1  # Periphery

if 'area' in df.columns:
    df['Region_category'] = df['area'].apply(get_region_category)
else:
    df['Region_category'] = 1

if 'location' in df.columns:
    df['Location_category'] = df['location'].apply(get_location_category)
else:
    df['Location_category'] = 1

df['RegionLocation'] = df['Region_category'] * df['Location_category']

print("✓ Region and location categories added")
print(f"Region categories: {df['Region_category'].value_counts().to_dict()}")
print(f"Location categories: {df['Location_category'].value_counts().to_dict()}")

## Step 5: Calculate Mode Score

Service & hierarchy of modes score (before normalization)

In [ ]:
def count_positive_mode_lines(row):
    """Count how many modes have lines."""
    count = 0
    mode_line_cols = ['BRT Lines', 'Cable Line Lines', 'Funicular Lines',
                      'HighSpeed Rail Lines', 'Interurban Rail Lines', 'LRT Lines',
                      'Metro Lines', 'Suburban Rail Lines']
    for col in mode_line_cols:
        if col in row.index and row[col] > 0:
            count += 1
    return count

def get_mode_score(row):
    """Calculate mode service score with diminishing returns and diversity bonus."""
    score = 0.0
    alpha = 0.1  # Diversity bonus factor
    
    for mode, weight in MODE_WEIGHTS.items():
        column_name = f'{mode} Lines'
        if column_name in row.index and row[column_name] > 0:
            # No diminishing returns in original notebook - just multiply
            score += row[column_name] * weight
    
    # Apply diversity bonus
    n_modes = row.get('Num_Modes', 1)
    score = score * (1 + alpha * (n_modes - 1))
    
    return score

# Count number of modes
df['Num_Modes'] = df.apply(count_positive_mode_lines, axis=1)

# Calculate mode score
df['score'] = df.apply(get_mode_score, axis=1)

print("✓ Mode scores calculated")
print(f"Score range: {df['score'].min():.2f} - {df['score'].max():.2f}")
print(f"Mean score: {df['score'].mean():.2f}")

## Step 6: Bus Terminal Score

In [ ]:
def bus_terminal_score(term_type):
    """Convert terminal type to score."""
    if pd.isna(term_type):
        return 0
    term_type = str(term_type).strip()
    
    if 'חניון לילה' in term_type:
        return 1
    elif 'מסוף קטן' in term_type or 'מסוף בינוני' in term_type:
        return 2
    elif 'מסוף גדול' in term_type or 'מתקן משולב' in term_type:
        return 3
    return 0

if 'term_type' in df.columns:
    df['bus_terminal'] = df['term_type'].apply(bus_terminal_score)
else:
    df['bus_terminal'] = 0

print("✓ Bus terminal scores added")
print(f"Terminal score distribution:\n{df['bus_terminal'].value_counts().sort_index()}")

## Step 7: Filter Eligible Hubs

Remove hubs with <1,000 demand OR only 1 mode

In [ ]:
print(f"Before filtering: {len(df)} hubs")

# Filter: TotalDemand >= 1000
df = df[df['TotalDemand'] >= 1000].copy()
print(f"After demand filter (≥1000): {len(df)} hubs")

# Filter: At least 2 modes
df = df[df['Mode_Planned'].apply(lambda x: len(x) > 1)].copy()
print(f"After mode filter (≥2 modes): {len(df)} hubs")

print("\n✓ Eligibility filtering complete")

## Step 8: Classify Hub Types

Using the logic from Group_n_Filter_Hubs notebook

In [ ]:
# Define hub type classification function
def classify_hub_tier(total_demand, modes, num_lines):
    """
    Classify hub into hierarchy tier based on modes, lines, and ridership.
    
    Based on Group_n_Filter_Hubs.ipynb logic.
    """
    if not isinstance(modes, list):
        modes = [modes] if pd.notna(modes) else []
    
    # Check for each mode type
    has_highspeed = 'HighSpeed Rail' in modes or 'HighSpeed' in modes
    has_interurban = 'Interurban Rail' in modes or 'Interurban' in modes
    has_suburban = 'Suburban Rail' in modes or 'Suburban' in modes
    has_metro = 'Metro' in modes
    has_brt = 'BRT' in modes
    has_lrt = 'LRT' in modes
    has_rail = 'Rail' in modes
    
    # National: (HighSpeed OR Interurban) AND ≥3 lines AND ≥50k demand
    if (has_highspeed or has_interurban or (has_rail and num_lines >= 3)) and (num_lines >= 3) and (total_demand >= 50000):
        return TIER_NATIONAL
    
    # Regional: (Suburban OR Metro OR Interurban OR HighSpeed) AND ≥3 lines
    elif (has_suburban or has_metro or has_interurban or has_highspeed or has_rail) and (num_lines >= 3):
        return TIER_METRO
    
    # Local: (BRT OR LRT) AND ≥3 lines AND ≥1000 demand
    elif (has_brt or has_lrt) and (num_lines >= 3) and (total_demand >= 1000):
        return TIER_LOCAL
    
    # Train Station: Has rail modes but ≤2 lines
    elif (has_interurban or has_highspeed or has_suburban or has_rail) and (num_lines <= 2):
        return 'Train Station'
    
    else:
        return 'Not Hub'

# Add Total_Unique_Lines if not present
if 'Total_Unique_Lines' not in df.columns:
    df['Total_Unique_Lines'] = df['Line_Unique'].apply(lambda x: len(x) if isinstance(x, list) else 0)

# Classify hub types
df['HubType'] = df.apply(
    lambda row: classify_hub_tier(
        total_demand=row['TotalDemand'],
        modes=row['Mode_Planned'],
        num_lines=row['Total_Unique_Lines']
    ),
    axis=1
)

print("✓ Hub types classified")
print("\nHub type distribution:")
print(df['HubType'].value_counts())

## Step 9: Normalize Scores by Hub Type

Normalize RegionLocation, score, and bus_terminal to 1-10 scale, separately per hub type

In [ ]:
def normalize_col_by_type(df, col, hub_type_col='HubType'):
    """Normalize a column to 1-10 scale, separately per hub type."""
    normalized_col = col + '_Norm'
    
    for hub_type in df[hub_type_col].unique():
        mask = df[hub_type_col] == hub_type
        values = df.loc[mask, col]
        
        min_val = values.min()
        max_val = values.max()
        
        if max_val > min_val:
            df.loc[mask, normalized_col] = 1 + (values - min_val) * 9 / (max_val - min_val)
        else:
            df.loc[mask, normalized_col] = 5.5  # Mid-point if all values are the same
    
    return df

# Normalize categorical scores
for col in ['RegionLocation', 'score', 'bus_terminal']:
    df = normalize_col_by_type(df, col)

print("✓ Categorical scores normalized")
print(f"\nNormalized score ranges:")
for col in ['RegionLocation_Norm', 'score_Norm', 'bus_terminal_Norm']:
    print(f"  {col}: {df[col].min():.2f} - {df[col].max():.2f}")

## Step 10: Normalize Demand Score

Apply log10 transformation and normalize per hub type

In [ ]:
def normalize_demand_by_type(df, demand_col='TotalDemand', hub_type_col='HubType'):
    """Normalize demand with log10 transformation, per hub type."""
    # Apply log10
    df['LogDemand'] = df[demand_col].apply(lambda x: 0 if x == 0 else np.log10(x))
    
    # Normalize per hub type
    for hub_type in df[hub_type_col].unique():
        mask = df[hub_type_col] == hub_type
        values = df.loc[mask, 'LogDemand']
        
        # Filter out zeros
        values_nonzero = values[values > 0]
        
        if len(values_nonzero) > 0:
            min_val = values_nonzero.min()
            max_val = values_nonzero.max()
            
            if max_val > min_val:
                df.loc[mask, 'TotalDemand_Norm'] = 1 + (values - min_val) * 9 / (max_val - min_val)
            else:
                df.loc[mask, 'TotalDemand_Norm'] = 5.5
        else:
            df.loc[mask, 'TotalDemand_Norm'] = 1.0
    
    return df

df = normalize_demand_by_type(df)

print("✓ Demand scores normalized")
print(f"TotalDemand_Norm range: {df['TotalDemand_Norm'].min():.2f} - {df['TotalDemand_Norm'].max():.2f}")

## Step 11: Calculate Population & Employment Score

Weighted by distance decay with β=1.5

In [ ]:
def calculate_pop_emp_score_by_type(df, hub_type_col='HubType', beta=1.5):
    """Calculate population/employment score with distance decay."""
    # Ring midpoints and decay
    radii = [0, 500, 1000, 1500]
    mids = np.array([(radii[i] + radii[i+1]) / 2 for i in range(len(radii)-1)])
    decay = mids ** beta
    
    pop_cols = ['pop_0_500', 'pop_500_1000', 'pop_1000_1500']
    emp_cols = ['emp_0_500', 'emp_500_1000', 'emp_1000_1500']
    
    # Check if columns exist
    if not all(col in df.columns for col in pop_cols + emp_cols):
        print("Warning: Pop/Emp columns not found, setting score to 5.0")
        df['PopEmp_Score_Raw'] = 5.0
        df['PopEmp_Score_Norm'] = 5.0
        return df
    
    scores = []
    for idx, row in df.iterrows():
        pop = np.array([row.get(col, 0) for col in pop_cols], dtype=float)
        emp = np.array([row.get(col, 0) for col in emp_cols], dtype=float)
        
        # Weight by hub type
        hub_type = row[hub_type_col]
        if hub_type in ['National', 'Regional', TIER_NATIONAL, TIER_METRO]:
            w_pop, w_emp = 0.2, 0.8
        else:
            w_pop, w_emp = 0.8, 0.2
        
        combined = w_pop * pop + w_emp * emp
        score = (combined / decay).sum()
        scores.append(score)
    
    df['PopEmp_Score_Raw'] = scores
    
    # Normalize per hub type
    for hub_type in df[hub_type_col].unique():
        mask = df[hub_type_col] == hub_type
        values = df.loc[mask, 'PopEmp_Score_Raw']
        
        min_val = values.min()
        max_val = values.max()
        
        if max_val > min_val:
            df.loc[mask, 'PopEmp_Score_Norm'] = 1 + (values - min_val) * 9 / (max_val - min_val)
        else:
            df.loc[mask, 'PopEmp_Score_Norm'] = 5.5
    
    return df

df = calculate_pop_emp_score_by_type(df)

print("✓ Pop/Emp scores calculated")
print(f"PopEmp_Score_Norm range: {df['PopEmp_Score_Norm'].min():.2f} - {df['PopEmp_Score_Norm'].max():.2f}")

## Step 12: Monte Carlo Scoring

Run 10,000 iterations with random weights (max 0.5 per criterion)

In [ ]:
def monte_carlo_scoring(df, hub_type_col='HubType', n_iterations=10000, random_seed=42):
    """Monte Carlo scoring with random weights."""
    scoring_cols = ['RegionLocation_Norm', 'bus_terminal_Norm', 'score_Norm', 
                    'TotalDemand_Norm', 'PopEmp_Score_Norm']
    
    # Check all columns exist
    missing = [col for col in scoring_cols if col not in df.columns]
    if missing:
        raise ValueError(f"Missing score columns: {missing}")
    
    np.random.seed(random_seed)
    results = []
    
    for hub_type in df[hub_type_col].unique():
        hub_df = df[df[hub_type_col] == hub_type].copy()
        hub_df['Sum_Simulated_Scores'] = 0.0
        
        for i in range(n_iterations):
            # Generate random weights (max 0.5 per criterion)
            weights = np.random.rand(len(scoring_cols))
            while any(weights > 0.5):
                weights = np.random.rand(len(scoring_cols))
            weights /= weights.sum()  # Normalize to sum to 1
            
            # Calculate weighted scores
            hub_df['Sum_Simulated_Scores'] += (hub_df[scoring_cols] * weights).sum(axis=1)
        
        # Average across iterations
        hub_df['Average_Simulated_Score'] = hub_df['Sum_Simulated_Scores'] / n_iterations
        hub_df = hub_df.drop(columns='Sum_Simulated_Scores')
        
        results.append(hub_df)
    
    return pd.concat(results, ignore_index=False).sort_index()

print("Running Monte Carlo simulation...")
df_scored = monte_carlo_scoring(df)

print("\n✓ Monte Carlo scoring complete")
print(f"\nFinal scores:")
print(df_scored['Average_Simulated_Score'].describe())

## Step 13: Add Rankings

In [ ]:
# Rank within hub type
df_scored['Rank_within_HubType'] = df_scored.groupby('HubType')['Average_Simulated_Score'].rank(
    method='dense', ascending=False
)

# Overall rank
df_scored['Overall_Rank'] = df_scored['Average_Simulated_Score'].rank(
    method='dense', ascending=False
)

print("✓ Rankings added")
print("\nTop 10 hubs overall:")
top_10 = df_scored.nlargest(10, 'Average_Simulated_Score')
display(top_10[['group', 'HubType', 'TotalDemand', 'Average_Simulated_Score', 'Overall_Rank', 'address']].head(10))

## Step 14: Results Summary by Hub Type

In [ ]:
print("Results by Hub Type:")
print("="*80)

for hub_type in df_scored['HubType'].unique():
    hub_data = df_scored[df_scored['HubType'] == hub_type]
    print(f"\n{hub_type}: {len(hub_data)} hubs")
    print(f"  Score range: {hub_data['Average_Simulated_Score'].min():.3f} - {hub_data['Average_Simulated_Score'].max():.3f}")
    print(f"  Mean score: {hub_data['Average_Simulated_Score'].mean():.3f}")
    print(f"  Median score: {hub_data['Average_Simulated_Score'].median():.3f}")
    
    # Top 5 in this type
    top_5 = hub_data.nsmallest(5, 'Rank_within_HubType')
    print(f"\n  Top 5 {hub_type} hubs:")
    for idx, row in top_5.iterrows():
        print(f"    {int(row['Rank_within_HubType'])}. Hub {row.get('group', idx)}: {row['Average_Simulated_Score']:.3f} ({row.get('address', 'N/A')[:50]}...)")

## Step 15: Export Results

In [ ]:
# Export to CSV
output_file = RESULTS_DIR / f"scored_hubs_{pd.Timestamp.now().strftime('%Y%m%d')}.csv"
df_scored.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"✓ Results exported to: {output_file}")
print(f"\nTotal hubs scored: {len(df_scored)}")
print(f"Columns in output: {len(df_scored.columns)}")

## Summary

Pipeline complete! The results include:

- **Hub classification** (National, Regional, Local, Train Station)
- **5 normalized scoring criteria** (1-10 scale per hub type):
  1. RegionLocation_Norm
  2. bus_terminal_Norm
  3. score_Norm (mode service score)
  4. TotalDemand_Norm (log-transformed)
  5. PopEmp_Score_Norm
- **Monte Carlo aggregated score** (average of 10,000 random weight iterations)
- **Rankings** (overall and within hub type)

All scores are normalized separately per hub type for fair comparison within tiers.